In [1]:
import requests
import csv

def human_readable_bytes(num_bytes, decimals=2):
    if num_bytes == 0:
        return "0 B"
    units = ["B", "KB", "MB", "GB", "TB"]
    idx = 0
    size = float(num_bytes)
    while size >= 1024 and idx < len(units)-1:
        size /= 1024.0
        idx += 1
    return f"{size:.{decimals}f} {units[idx]}"

def fetch_and_write_csv(api_key, output_csv="../../data/csv_files/materialized_views.csv"):
    api_url = f"https://api.dune.com/api/v1/materialized-views?api_key={api_key}"
    resp = requests.get(api_url)
    resp.raise_for_status()
    data = resp.json()
    mv_list = data.get("materialized_views", [])

    with open(output_csv, mode="w", newline='', encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "sql_id", "query_id", "is_private", "table_size_bytes", "table_size_human"])
        for mv in mv_list:
            raw_bytes = mv.get("table_size_bytes", 0)
            human = human_readable_bytes(raw_bytes)
            writer.writerow([
                mv.get("id"),
                mv.get("sql_id"),
                mv.get("query_id"),
                mv.get("is_private"),
                raw_bytes,
                human
            ])
    print(f"Wrote {len(mv_list)} records to {output_csv}")

if __name__ == "__main__":
    YOUR_API_KEY = "TwBLHAAyy9Ec86yp5x5nWVcZu9XKozTn"  # use your key
    fetch_and_write_csv(YOUR_API_KEY)


Wrote 247 records to materialized_views.csv


In [2]:
import csv

input_file = "../../data/csv_files/materialized_views.csv"
output_file = "../../data/csv_files/materialized_views_sorted.csv"

rows = []

# Read the CSV
with open(input_file, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Convert the bytes column back to integer for sorting
        row["table_size_bytes"] = int(row["table_size_bytes"])
        rows.append(row)

# Sort in descending order of size
rows_sorted = sorted(rows, key=lambda x: x["table_size_bytes"], reverse=True)

# Write the new sorted CSV
with open(output_file, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=rows_sorted[0].keys())
    writer.writeheader()
    writer.writerows(rows_sorted)

print(f"Sorted file created → {output_file}")


Sorted file created → materialized_views_sorted.csv
